In [6]:
import re
import os
import numpy as np
from PIL import Image, UnidentifiedImageError

In [7]:
IMG_EXT_PATTERN = re.compile(r'^.+\.(png|jpg|jpeg|bmp)$', re.IGNORECASE)
def load_image(filename: str) -> np.ndarray:
    if not isinstance(filename, str) or not filename.strip():
        raise ValueError("Имя файла пустое или не строка")
    if not IMG_EXT_PATTERN.match(filename):
        raise ValueError(f"Ошибочный формат файла: '{filename}' — ожидается .png/.jpg/.jpeg/.bmp")
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Файл не существует: {filename}")
    if not os.path.isfile(filename):
        raise IsADirectoryError(f"Путь не является файлом: {filename}")
    if not os.access(filename, os.R_OK):
        raise PermissionError(f"Нет прав на чтение: {filename}")
    try:
        with Image.open(filename) as img:
            img.verify()
        with Image.open(filename) as img:
            arr = np.array(img)
    except (UnidentifiedImageError, OSError) as e:
        raise ValueError(f"Повреждённый или неподдерживаемый файл: {filename}") from e
    except MemoryError as e:
        raise MemoryError(f"Не хватает памяти для загрузки: {filename}") from e
    h, w = arr.shape[:2]
    if h == 0 or w == 0:
        raise ValueError(f"Изображение пустое: {filename}")
    if arr.dtype != np.uint8 or arr.min() < 0 or arr.max() > 255:
        raise ValueError("Недопустимые значения пикселей (ожидается uint8 0..255)")
    return arr

In [11]:
Image.fromarray((np.random.rand(32, 48, 3) * 255).astype("uint8")).save("correct_img.png") # Корректное цветное PNG
Image.fromarray((np.random.rand(32, 48) * 255).astype("uint8")).save("gray.jpg") # Корректный ч/б JPG
Image.fromarray((np.random.rand(16, 16, 3) * 255).astype("uint8")).save("pic.bmp") # Корректный BMP
with open("fake.png", "wb") as f:
    f.write(b"not an image at all")
with open("data.json", "w") as f:
    f.write("{}")

In [12]:
tests = [
    ("correct_img.png",       None),
    ("gray.jpg",       None),
    ("pic.bmp",        None),
    ("fake.png",       ValueError),
    ("data.json",      ValueError),
    ("missing.png",    FileNotFoundError),
    (".",              ValueError),
]

for name, expected in tests:
    try:
        arr = load_image(name)
        result = f"OK shape={arr.shape} dtype={arr.dtype}"
        ok = expected is None
    except Exception as e:
        result = f"{type(e).__name__}: {e}"
        ok = expected is not None and isinstance(e, expected)
    print(("PASS" if ok else "FAIL"), name, "->", result)

PASS correct_img.png -> OK shape=(32, 48, 3) dtype=uint8
PASS gray.jpg -> OK shape=(32, 48) dtype=uint8
PASS pic.bmp -> OK shape=(16, 16, 3) dtype=uint8
PASS fake.png -> ValueError: Повреждённый или неподдерживаемый файл: fake.png
PASS data.json -> ValueError: Ошибочный формат файла: 'data.json' — ожидается .png/.jpg/.jpeg/.bmp
PASS missing.png -> FileNotFoundError: Файл не существует: missing.png
PASS . -> ValueError: Ошибочный формат файла: '.' — ожидается .png/.jpg/.jpeg/.bmp
